In [20]:
# Imports and Configuration
import pandas as pd
import numpy as np
import re

# Set random seed for reproducibility
np.random.seed(42)

# Configure pandas for future compatibility
pd.set_option('future.no_silent_downcasting', True)

# Load dataset
df = pd.read_csv('ad_campaign_performance.csv')
df_clean = df.copy()  # Preserve original for reference

# Initial overview
print("Initial Data Overview:")
print(df_clean.info())
print("\nSample Data:")
print(df_clean.head())

# Save original for reproducibility
df.to_csv('original_ad_campaign_performance.csv', index=False)

Initial Data Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   AdID          1020 non-null   object 
 1   CampaignName  1020 non-null   object 
 2   Platform      1020 non-null   object 
 3   Date          1020 non-null   object 
 4   Impressions   1020 non-null   int64  
 5   Clicks        1020 non-null   int64  
 6   Cost          1020 non-null   object 
 7   Conversions   929 non-null    float64
 8   CTR           927 non-null    float64
dtypes: float64(2), int64(2), object(5)
memory usage: 71.8+ KB
None

Sample Data:
     AdID   CampaignName  Platform        Date  Impressions  Clicks  \
0  AD1001    SpringPromo    Google  2024-01-07         1080      89   
1  AD1002     WinterSale      Meta  2024-01-29          537       7   
2  AD1003    SpringPromo  LinkedIn  2024-01-06         3770     361   
3  AD1004     WinterSale  Linked

## Task 1: Unify Dates
Convert `Date` to 'YYYY-MM-DD', normalizing valid formats (YYYY-MM-DD, MM/DD/YYYY) and fixing invalid US dates (e.g., '2024-13-01' → '01/13/2024') in ~1,020 rows. Approximately 5% of rows (~50) have invalid or MM/DD/YYYY formats. Standardization ensures consistent date analysis for ad campaign performance (e.g., daily click trends).

In [22]:
# Task 1: Unify Dates

# Preserve original Date for reference
df_clean['OriginalDate'] = df_clean['Date']

# Function to normalize date formats
def normalize_date(timestamp):
    """Normalize MM/DD/YYYY or YYYY-MM-DD to YYYY-MM-DD, fix invalid months."""
    if pd.isna(timestamp):
        return timestamp
    # Try MM/DD/YYYY
    match_us = re.match(r'(\d{2})/(\d{2})/(\d{4})$', timestamp)
    if match_us:
        month, day, year = match_us.groups()
        return f'{year}-{month}-{day}'
    # Try YYYY-MM-DD with invalid month
    match_invalid = re.match(r'(\d{4})-(\d{2})-(\d{2})$', timestamp)
    if match_invalid:
        year, month, day = match_invalid.groups()
        if int(month) > 12:  # Fix US format (e.g., '2024-13-01' → '2024-01-13')
            return f'{year}-{day}-{month}'
        return timestamp
    return timestamp

# Apply normalization and parse to datetime
df_clean['Date'] = df_clean['Date'].apply(normalize_date)
df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce', format='%Y-%m-%d')

# Drop rows with NaT in Date
pre_drop_rows = len(df_clean)
df_clean = df_clean[df_clean['Date'].notna()].copy().reset_index(drop=True)

# Log changes
print(f"Rows before Task 1: {pre_drop_rows}")
print(f"Rows dropped due to NaT: {pre_drop_rows - len(df_clean)}")
print(f"Rows after Task 1: {len(df_clean)}")
print(f"Original Date sample:\n{df_clean['OriginalDate'].sample(5, random_state=42)}")
print(f"Date dtype: {df_clean['Date'].dtype}")

Rows before Task 1: 1020
Rows dropped due to NaT: 0
Rows after Task 1: 1020
Original Date sample:
523    2024-01-25
602    2024-01-08
526    01/24/2024
31     01/10/2024
616    01/19/2024
Name: OriginalDate, dtype: object
Date dtype: datetime64[ns]


In [28]:
# Verify Task 1
print("Task 1 Verification:")
# Check date format (string representation)
pattern = r'^\d{4}-\d{2}-\d{2}$'
valid_dates = df_clean['Date'].dt.strftime('%Y-%m-%d').str.match(pattern, na=False)
print(f"Invalid Date formats: {(~valid_dates).sum()}")
if (~valid_dates).any():
    print(f"Invalid examples:\n{df_clean[~valid_dates][['Date']].head()}")

# Validate dates (no NaT expected after drop)
invalid_dates = df_clean['Date'].isna()
print(f"Invalid dates (NaT): {invalid_dates.sum()}")

# Sample cleaned dates
print("\nSample of standardized Dates:")
print(df_clean['Date'].sample(5, random_state=42))

Task 1 Verification:
Invalid Date formats: 0
Invalid dates (NaT): 0

Sample of standardized Dates:
523   2024-01-25
602   2024-01-08
526   2024-01-24
31    2024-01-10
616   2024-01-19
Name: Date, dtype: datetime64[ns]


## Task 2: Standardize Platform
Normalize `Platform` to consistent names ('Google', 'Meta', 'LinkedIn') in ~1,020 rows (post-Task 1). Approximately 5% of rows (~50) have inconsistent formats (e.g., 'FB', 'Facebook', 'google'). Standardization ensures accurate grouping for ad campaign metrics (e.g., clicks by platform).

In [31]:
# Task 2: Standardize Platform

# Preserve original Platform for reference
df_clean['OriginalPlatform'] = df_clean['Platform']

# Define mapping for normalization
platform_map = {
    'Google': 'Google',
    'google': 'Google',
    'GOOGLE': 'Google',
    'Meta': 'Meta',
    'meta': 'Meta',
    'META': 'Meta',
    'FB': 'Meta',
    'Facebook': 'Meta',
    'facebook': 'Meta',
    'LinkedIn': 'LinkedIn',
    'linkedin': 'LinkedIn',
    'LINKEDIN': 'LinkedIn'
}

# Normalize Platform
df_clean['Platform'] = df_clean['Platform'].replace(platform_map)

# Log changes
print(f"Rows before Task 2: {len(df_clean)}")
print(f"Original Platform values:\n{df_clean['OriginalPlatform'].value_counts(dropna=False)}")

Rows before Task 2: 1020
Original Platform values:
OriginalPlatform
Google      343
Meta        326
LinkedIn    307
FB           16
google       12
Facebook      9
meta          4
linkedin      3
Name: count, dtype: int64


In [33]:
# Verify Task 2
print("Task 2 Verification:")
# Check for valid Platform values
valid_platforms = df_clean['Platform'].isin(['Google', 'Meta', 'LinkedIn'])
print(f"Invalid Platform values: {(~valid_platforms).sum()}")
if (~valid_platforms).any():
    print(f"Invalid examples:\n{df_clean[~valid_platforms][['Platform']].head()}")

# Sample Platform values
print("\nSample of standardized Platforms:")
print(df_clean['Platform'].sample(5, random_state=42))

# Check for NaN
print(f"NaN in Platform: {df_clean['Platform'].isna().sum()}")

# Final Platform counts
print(f"Final Platform values:\n{df_clean['Platform'].value_counts()}")

Task 2 Verification:
Invalid Platform values: 0

Sample of standardized Platforms:
523    LinkedIn
602      Google
526      Google
31         Meta
616    LinkedIn
Name: Platform, dtype: object
NaN in Platform: 0
Final Platform values:
Platform
Google      355
Meta        355
LinkedIn    310
Name: count, dtype: int64


## Task 3: Clean Cost
Convert `Cost` to float, removing text prefixes (e.g., '$100.50', 'USD100') and dropping rows with NaN or negative values in ~1,020 rows (post-Task 2). Approximately 5% (~50) have text or negative values. Dropping ensures accurate ad spend analysis (e.g., cost per click).

In [36]:
# Task 3: Clean Cost

# Preserve original Cost
df_clean['OriginalCost'] = df_clean['Cost']

# Function to clean Cost
def clean_cost(value):
    """Convert Cost to float, remove text prefixes, return NaN for invalid."""
    if pd.isna(value):
        return np.nan
    try:
        value_str = str(value).replace('$', '').replace('USD', '')
        return float(value_str)
    except (ValueError, TypeError):
        return np.nan

# Apply cleaning
df_clean['Cost'] = df_clean['Cost'].apply(clean_cost)

# Drop rows with NaN or negative Cost
pre_drop_rows = len(df_clean)
df_clean = df_clean[df_clean['Cost'].notna() & (df_clean['Cost'] >= 0)].copy().reset_index(drop=True)

# Log changes
print(f"Rows before Task 3: {pre_drop_rows}")
print(f"Rows dropped (NaN or negative Cost): {pre_drop_rows - len(df_clean)}")
print(f"Rows after Task 3: {len(df_clean)}")
print(f"Original Cost sample:\n{df_clean['OriginalCost'].sample(5, random_state=42)}")
print(f"Cost dtype: {df_clean['Cost'].dtype}")

Rows before Task 3: 1020
Rows dropped (NaN or negative Cost): 17
Rows after Task 3: 1003
Original Cost sample:
626     313.3968765444286
629     89.61574469987515
847    255.19594621237061
514      410.399296181964
365     386.1414061151268
Name: OriginalCost, dtype: object
Cost dtype: float64


In [38]:
# Verify Task 3
print("Task 3 Verification:")
# Check for non-numeric or NaN
invalid_costs = df_clean['Cost'].isna() | ~df_clean['Cost'].apply(lambda x: isinstance(x, (int, float)))
print(f"Invalid Cost values: {invalid_costs.sum()}")

# Check for negative or zero values
print(f"Negative or zero Cost values: {(df_clean['Cost'] <= 0).sum()}")

# Sample cleaned Costs
print("\nSample of cleaned Costs:")
print(df_clean['Cost'].sample(5, random_state=42))

# Check range
print(f"Cost range: min={df_clean['Cost'].min()}, max={df_clean['Cost'].max()}")
print(f"NaN in Cost: {df_clean['Cost'].isna().sum()}")

Task 3 Verification:
Invalid Cost values: 0
Negative or zero Cost values: 0

Sample of cleaned Costs:
626    313.396877
629     89.615745
847    255.195946
514    410.399296
365    386.141406
Name: Cost, dtype: float64
Cost range: min=50.237664292367015, max=499.62478014241697
NaN in Cost: 0


## Task 4: Fix Impressions/Clicks
Remove rows with zero `Impressions` but non-zero `Clicks` in ~970 rows (post-Task 3). Approximately 2% of rows (~20) have this invalid condition due to logging errors. Dropping ensures accurate ad campaign metrics (e.g., click-through rate).

In [41]:
# Task 4: Fix Impressions/Clicks

# Preserve original Impressions and Clicks
df_clean['OriginalImpressions'] = df_clean['Impressions']
df_clean['OriginalClicks'] = df_clean['Clicks']

# Identify invalid rows (zero Impressions, non-zero Clicks)
invalid_rows = (df_clean['Impressions'] == 0) & (df_clean['Clicks'] > 0)

# Drop invalid rows
pre_drop_rows = len(df_clean)
df_clean = df_clean[~invalid_rows].copy().reset_index(drop=True)

# Log changes
print(f"Rows before Task 4: {pre_drop_rows}")
print(f"Rows dropped (zero Impressions, non-zero Clicks): {invalid_rows.sum()}")
print(f"Rows after Task 4: {len(df_clean)}")
print(f"Original Impressions/Clicks sample:\n{df_clean[['OriginalImpressions', 'OriginalClicks']].sample(5, random_state=42)}")

Rows before Task 4: 1003
Rows dropped (zero Impressions, non-zero Clicks): 13
Rows after Task 4: 990
Original Impressions/Clicks sample:
     OriginalImpressions  OriginalClicks
213                 3875             213
331                 1520             106
501                 4534             192
309                 3701             330
88                  4844             449


In [43]:
# Verify Task 4
print("Task 4 Verification:")
# Check for remaining invalid rows
invalid_rows = (df_clean['Impressions'] == 0) & (df_clean['Clicks'] > 0)
print(f"Remaining invalid rows (zero Impressions, non-zero Clicks): {invalid_rows.sum()}")

# Check for NaN
print(f"NaN in Impressions: {df_clean['Impressions'].isna().sum()}")
print(f"NaN in Clicks: {df_clean['Clicks'].isna().sum()}")

# Sample cleaned data
print("\nSample of cleaned Impressions/Clicks:")
print(df_clean[['Impressions', 'Clicks']].sample(5, random_state=42))

# Check data range
print(f"Impressions range: min={df_clean['Impressions'].min()}, max={df_clean['Impressions'].max()}")
print(f"Clicks range: min={df_clean['Clicks'].min()}, max={df_clean['Clicks'].max()}")

Task 4 Verification:
Remaining invalid rows (zero Impressions, non-zero Clicks): 0
NaN in Impressions: 0
NaN in Clicks: 0

Sample of cleaned Impressions/Clicks:
     Impressions  Clicks
213         3875     213
331         1520     106
501         4534     192
309         3701     330
88          4844     449
Impressions range: min=503, max=4989
Clicks range: min=0, max=470


## Task 5: Resolve Duplicates
Remove exact duplicate `AdID` entries in ~970 rows (post-Task 4). Approximately 2% of rows (~20) are duplicates due to logging errors. Different `AdID` values for the same `CampaignName`, `Platform`, `Date` are valid (distinct ads). Deduplication ensures accurate ad campaign metrics (e.g., unique ad performance).

In [50]:
# Task 5: Resolve Duplicates

# Preserve original indices
df_clean['OriginalIndex'] = df_clean.index

# Drop exact AdID duplicates and save dropped rows
df_pre_dedup = df_clean.copy()
df_clean = df_clean.drop_duplicates(subset=['AdID'], keep='first').copy().reset_index(drop=True)
dropped_exact = df_pre_dedup[df_pre_dedup['AdID'].duplicated(keep='first')]
dropped_exact.to_csv('dropped_exact_duplicates_task5.csv', index=False)

# Log changes
print(f"Rows before Task 5: {len(df_pre_dedup)}")
print(f"Exact AdID duplicates removed: {len(dropped_exact)}")
print(f"Rows after Task 5: {len(df_clean)}")

Rows before Task 5: 990
Exact AdID duplicates removed: 20
Rows after Task 5: 970


In [52]:
# Verify Task 5
print("Task 5 Verification:")
# Check index continuity
print(f"Index is sequential: {df_clean.index.equals(pd.RangeIndex(start=0, stop=len(df_clean)))}")

# Check for exact AdID duplicates
print(f"Remaining AdID duplicates: {df_clean['AdID'].duplicated().sum()}")

# Sample data
print("\nSample of cleaned data:")
print(df_clean[['AdID', 'CampaignName', 'Platform', 'Date']].sample(5, random_state=42))

# Check for NaN in key columns
print(f"NaN in AdID: {df_clean['AdID'].isna().sum()}")
print(f"NaN in CampaignName: {df_clean['CampaignName'].isna().sum()}")
print(f"NaN in Platform: {df_clean['Platform'].isna().sum()}")
print(f"NaN in Date: {df_clean['Date'].isna().sum()}")

# Keep OriginalIndex for potential audit in Task 6

Task 5 Verification:
Index is sequential: True
Remaining AdID duplicates: 0

Sample of cleaned data:
       AdID   CampaignName  Platform       Date
575  AD1591  ProductLaunch      Meta 2024-01-13
259  AD1269  ProductLaunch  LinkedIn 2024-01-18
388  AD1402     WinterSale  LinkedIn 2024-01-23
495  AD1510  ProductLaunch      Meta 2024-01-13
70   AD1074  ProductLaunch  LinkedIn 2024-01-28
NaN in AdID: 0
NaN in CampaignName: 0
NaN in Platform: 0
NaN in Date: 0


## Task 6: Fill Blanks
Fill blanks ('') and NaN in `Conversions` and `CTR` with 0 in ~950 rows (post-Task 5). Approximately 10% of rows (~95) have blank values, with possible NaN from prior tasks. Convert `Conversions` to int, `CTR` to float. Filling ensures accurate ad campaign metrics (e.g., conversion rates). Create a final DataFrame with cleaned columns only.

In [59]:
# Task 6: Fill Blanks

# Preserve original Conversions and CTR
df_clean['OriginalConversions'] = df_clean['Conversions']
df_clean['OriginalCTR'] = df_clean['CTR']

# Pre-cleaning check with regex
print("Pre-cleaning check:")
numeric_pattern = r'^-?\d*\.?\d+$'  # Matches integers or floats (e.g., "5", "0.05")
print(f"Conversions non-numeric or NaN/blank:\n{df_clean['Conversions'].apply(lambda x: pd.isna(x) or (isinstance(x, str) and x != '' and not re.match(numeric_pattern, x))).value_counts()}")
print(f"CTR non-numeric or NaN/blank:\n{df_clean['CTR'].apply(lambda x: pd.isna(x) or (isinstance(x, str) and x != '' and not re.match(numeric_pattern, x))).value_counts()}")

# Fill blanks and NaN with 0
df_clean['Conversions'] = df_clean['Conversions'].replace(['', np.nan], 0).astype(int)
df_clean['CTR'] = df_clean['CTR'].replace(['', np.nan], 0).astype(float)

# Log changes
print(f"\nRows before Task 6: {len(df_clean)}")
print(f"Original Conversions values:\n{df_clean['OriginalConversions'].value_counts(dropna=False)}")
print(f"Original CTR values:\n{df_clean['OriginalCTR'].value_counts(dropna=False)}")
print(f"Conversions dtype: {df_clean['Conversions'].dtype}")
print(f"CTR dtype: {df_clean['CTR'].dtype}")

Pre-cleaning check:
Conversions non-numeric or NaN/blank:
Conversions
False    882
True      88
Name: count, dtype: int64
CTR non-numeric or NaN/blank:
CTR
False    892
True      78
Name: count, dtype: int64

Rows before Task 6: 970
Original Conversions values:
OriginalConversions
NaN     88
13.0    53
8.0     53
19.0    52
10.0    49
1.0     47
0.0     47
14.0    47
4.0     43
7.0     42
12.0    42
5.0     41
17.0    40
18.0    39
16.0    38
9.0     37
3.0     37
15.0    37
2.0     37
6.0     36
20.0    36
11.0    29
Name: count, dtype: int64
Original CTR values:
OriginalCTR
NaN         78
0.000000     3
0.006536     2
0.042017     2
0.002933     2
            ..
0.040086     1
0.010325     1
0.051712     1
0.069737     1
0.018723     1
Name: count, Length: 885, dtype: int64
Conversions dtype: int64
CTR dtype: float64


In [61]:
# Verify Task 6
print("Task 6 Verification:")
# Check for blanks or NaN
print(f"Blank or NaN in Conversions: {(df_clean['Conversions'].isna() | (df_clean['Conversions'] == '')).sum()}")
print(f"Blank or NaN in CTR: {(df_clean['CTR'].isna() | (df_clean['CTR'] == '')).sum()}")

# Check valid types
invalid_conversions = ~df_clean['Conversions'].apply(lambda x: isinstance(x, int))
invalid_ctr = ~df_clean['CTR'].apply(lambda x: isinstance(x, (int, float)))
print(f"Invalid Conversions values: {invalid_conversions.sum()}")
print(f"Invalid CTR values: {invalid_ctr.sum()}")

# Sample data
print("\nSample of cleaned Conversions/CTR:")
print(df_clean[['Conversions', 'CTR']].sample(5, random_state=42))

# Final value ranges
print(f"Conversions range: min={df_clean['Conversions'].min()}, max={df_clean['Conversions'].max()}")
print(f"CTR range: min={df_clean['CTR'].min()}, max={df_clean['CTR'].max()}")

Task 6 Verification:
Blank or NaN in Conversions: 0
Blank or NaN in CTR: 0
Invalid Conversions values: 0
Invalid CTR values: 0

Sample of cleaned Conversions/CTR:
     Conversions       CTR
575            6  0.081074
259            3  0.000000
388            7  0.027406
495            7  0.043548
70             9  0.053099
Conversions range: min=0, max=20
CTR range: min=0.0, max=0.1


In [63]:
# Create final cleaned DataFrame
cleaned_cols = ['AdID', 'CampaignName', 'Platform', 'Date', 'Impressions', 
                'Clicks', 'Cost', 'Conversions', 'CTR']
df_final = df_clean[cleaned_cols].copy()

# Save final dataset
df_final.to_csv('cleaned_ad_campaign_performance.csv', index=False)

# Log final state
print("Final Cleaned DataFrame:")
print(df_final.info())
print("\nSample of final data:")
print(df_final.sample(5, random_state=42))

Final Cleaned DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 970 entries, 0 to 969
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   AdID          970 non-null    object        
 1   CampaignName  970 non-null    object        
 2   Platform      970 non-null    object        
 3   Date          970 non-null    datetime64[ns]
 4   Impressions   970 non-null    int64         
 5   Clicks        970 non-null    int64         
 6   Cost          970 non-null    float64       
 7   Conversions   970 non-null    int64         
 8   CTR           970 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 68.3+ KB
None

Sample of final data:
       AdID   CampaignName  Platform       Date  Impressions  Clicks  \
575  AD1591  ProductLaunch      Meta 2024-01-13         3577     290   
259  AD1269  ProductLaunch  LinkedIn 2024-01-18         1161     11